# Notebook 06 — Backtesting

**Goal:** Convert model predictions into trading signals and evaluate P&L under realistic market conditions.

Includes slippage, commission costs (0.5-2 bps), and per-regime performance breakdown.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.utils import set_seed, plot_equity_curve
from src.backtest import (
    backtest_strategy, performance_by_regime,
    compute_sharpe_ratio, compute_sortino_ratio, compute_max_drawdown
)
from src.features import compute_mid_price, compute_volatility_regime, compute_volatility

set_seed(42)
sns.set_theme(style='whitegrid', font_scale=1.1)
%matplotlib inline

RESULTS = Path('../results')

## 1. Load Predictions & Prices

In [ ]:
# Load predictions from all models
df = pd.read_parquet('../data/processed/features.parquet')
mid_prices = df['mid_price'].values
vol = df['volatility'].values
regimes = df['vol_regime'].values

# Test-set region (last 15%)
n = len(mid_prices)
test_start = int(n * 0.85)
test_prices = mid_prices[test_start:]
test_regimes = regimes[test_start:]

# Load saved predictions
lstm_preds = np.load('../data/processed/lstm_preds.npy')
gru_preds = np.load('../data/processed/gru_preds.npy')
tft_preds = np.load('../data/processed/tft_preds.npy')

# Also load LightGBM predictions (from baseline notebook)
import joblib
lgbm_model = joblib.load(RESULTS / 'models' / 'lightgbm_best.joblib')
feature_cols = [c for c in df.columns if not c.startswith('label')]
X_test_flat = np.nan_to_num(df[feature_cols].values[test_start:].astype(np.float32))
lgbm_preds = lgbm_model.predict(X_test_flat)

model_preds = {
    'LightGBM': lgbm_preds,
    'LSTM': lstm_preds,
    'GRU': gru_preds,
    'TFT': tft_preds,
}

print(f'Test set: {len(test_prices)} samples')
for name, preds in model_preds.items():
    print(f'  {name}: {len(preds)} predictions')

## 2. Run Backtests (Multiple Cost Scenarios)

In [ ]:
cost_scenarios = [
    {'name': 'Low Cost (0.5 bps)', 'commission': 0.5, 'slippage': 0.5},
    {'name': 'Base Cost (1 bps)', 'commission': 1.0, 'slippage': 0.5},
    {'name': 'High Cost (2 bps)', 'commission': 2.0, 'slippage': 1.0},
]

all_bt_results = []

for model_name, preds in model_preds.items():
    # Align predictions with test prices
    min_len = min(len(preds), len(test_prices))
    p = preds[:min_len]
    tp = test_prices[:min_len]
    
    for scenario in cost_scenarios:
        print(f'\n--- {model_name} | {scenario["name"]} ---')
        result = backtest_strategy(
            p, tp,
            commission_bps=scenario['commission'],
            slippage_bps=scenario['slippage']
        )
        all_bt_results.append({
            'Model': model_name,
            'Cost Scenario': scenario['name'],
            'Total Return': result['total_return'],
            'Sharpe': result['sharpe_ratio'],
            'Sortino': result['sortino_ratio'],
            'Max DD': result['max_drawdown'],
            'Win Rate': result['win_rate'],
            '# Trades': result['n_trades'],
        })

In [ ]:
bt_df = pd.DataFrame(all_bt_results)
bt_df.to_csv(RESULTS / 'tables' / 'backtest_results.csv', index=False)

print('\n=== Full Backtest Results ===')
print(bt_df.to_string(index=False))

## 3. Equity Curves (Base Cost)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
colors = {'LightGBM': '#2ecc71', 'LSTM': '#3498db', 'GRU': '#e67e22', 'TFT': '#9b59b6'}

for model_name, preds in model_preds.items():
    min_len = min(len(preds), len(test_prices))
    result = backtest_strategy(preds[:min_len], test_prices[:min_len],
                               commission_bps=1.0, slippage_bps=0.5)
    ax.plot(result['equity_curve'], label=model_name,
            color=colors[model_name], linewidth=1.5)

ax.set_title('Equity Curves — All Models (1 bps commission + 0.5 bps slippage)')
ax.set_xlabel('Time Step')
ax.set_ylabel('Portfolio Value ($)')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.axhline(100000, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
fig.savefig(RESULTS / 'plots' / 'equity_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Performance by Volatility Regime

In [ ]:
# Best model regime analysis
for model_name, preds in model_preds.items():
    min_len = min(len(preds), len(test_prices))
    result = backtest_strategy(preds[:min_len], test_prices[:min_len],
                               commission_bps=1.0, slippage_bps=0.5)
    
    regime_df = performance_by_regime(
        result['net_returns'],
        test_regimes[:len(result['net_returns'])]
    )
    print(f'\n=== {model_name} — Performance by Volatility Regime ===')
    print(regime_df.to_string(index=False))

# Save regime analysis for best model
regime_df.to_csv(RESULTS / 'tables' / 'regime_performance.csv', index=False)

## 5. Transaction Cost Sensitivity

In [ ]:
# Plot Sharpe ratio vs cost for each model
fig, ax = plt.subplots(figsize=(10, 6))

for model_name in model_preds:
    subset = bt_df[bt_df['Model'] == model_name]
    ax.plot(range(len(cost_scenarios)), subset['Sharpe'].values,
            marker='o', label=model_name, color=colors[model_name], linewidth=2)

ax.set_xticks(range(len(cost_scenarios)))
ax.set_xticklabels([s['name'] for s in cost_scenarios])
ax.set_ylabel('Sharpe Ratio')
ax.set_title('Sharpe Ratio Sensitivity to Transaction Costs')
ax.legend()
ax.grid(True, alpha=0.3)
ax.axhline(0, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
fig.savefig(RESULTS / 'plots' / 'cost_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

- All models backtested under 3 transaction cost scenarios
- Equity curves show cumulative P&L evolution  
- Regime analysis reveals which vol environments favor each model
- Cost sensitivity shows strategy robustness to friction

**Next:** Notebook 07 — SHAP Explainability